# CIFAR-10 Linear Classification 실습
이 노트북은 PyTorch를 사용한 CIFAR-10의 선형(로지스틱) 분류 실습입니다. 각 코드 셀에 한국어 주석을 달았고, 학습 흐름과 하이퍼파라미터(learning rate, batch size, weight decay, epochs)에 따른 트레이드오프 예시를 보여줍니다.

주의: 이 노트북은 교육 목적이며, 선형 모델은 CIFAR-10처럼 복잡한 이미지에 대해 성능이 제한적입니다. 그러나 모델 구조와 하이퍼파라미터 효과를 이해하기에 좋습니다.

## 1) 환경 설정 및 의존성
필요한 패키지: `torch`, `torchvision`, `matplotlib`, `tqdm`, `numpy`
만약 로컬에 설치되어 있지 않으면 아래 셀의 주석을 해제해 설치하세요.

In [ ]:
# !pip install torch torchvision matplotlib tqdm numpy  # 필요 시 주석 해제해서 실행
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

# 디바이스 설정: GPU가 있으면 사용, 없으면 CPU 사용
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 2) 데이터 불러오기
- CIFAR-10은 32x32 RGB 이미지 10개 클래스로 구성됩니다.
- 선형 분류기를 쓰려면 이미지를 평탄화(flatten)하여 입력 차원으로 사용합니다.
- Normalize는 평균/표준편차를 사용해 입력 스케일을 맞추기 위해 사용합니다.

In [ ]:
# 데이터 전처리: Tensor 변환 및 정규화
transform = transforms.Compose([
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261))

## 3) 선형 모델 정의
- 입력: 3*32*32 (RGB flattened)
- 출력: 10(클래스 수)
- 단순 선형 계층 하나로 구성됩니다 (로지스틱 회귀 스타일).

In [ ]:
class LinearCIFAR10(nn.Module):
    def __init__(self):
        super().__init__()
        # 이미지를 평탄화하여 바로 선형 계층에 넣음
        self.fc = nn.Linear(3*32*32, 10)
    def forward(self, x):
        # 배치 x 채널 x H x W -> 배치 x (채널*H*W)
        x = x.view(x.size(0), -1)
        return self.fc(x)

# 모델 생성 및 디바이스 이동
model = LinearCIFAR10().to(device)
print(model)

## 4) 학습/평가 함수 정의
- `train_one_epoch`: 한 epoch 동안 학습하고 손실을 반환
- `evaluate`: 테스트셋에서 정확도를 계산
- 주석으로 각 단계의 역할 설명

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    avg_loss = running_loss / len(dataloader.dataset)
    return avg_loss

def evaluate(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += targets.size(0)
            correct += (predicted == targets).sum().item()
    acc = correct / total
    return acc

## 5) 기본 학습 실행 (간단한 예시)
- 기본 하이퍼파라미터: learning rate=1e-2, batch_size=128, epochs=10, weight_decay=0
- 실제 학습 시간 절약을 위해 epochs는 작게 설정되어 있습니다. 필요시 늘려서 실험하세요.

In [ ]:
# 하이퍼파라미터
lr = 1e-2
batch_size = 128
epochs = 10
weight_decay = 0.0

trainloader = DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)

# 모델/옵티마이저/손실 재생성 (초기화)
model = LinearCIFAR10().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)

train_losses = []
test_accs = []
for epoch in range(epochs):
    loss = train_one_epoch(model, trainloader, optimizer, criterion, device)
    acc = evaluate(model, testloader, device)
    train_losses.append(loss)
    test_accs.append(acc)
    print(f'Epoch {epoch+1}/{epochs} - loss: {loss:.4f} - test_acc: {acc:.4f}')

# 간단한 결과 시각화
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(train_losses, marker='o')
plt.title('Train loss')
plt.subplot(1,2,2)
plt.plot(test_accs, marker='o')
plt.title('Test accuracy')
plt.show()

## 6) 하이퍼파라미터 트레이드오프 실험
- 학습률(learning rate), 배치 크기(batch size), weight decay(정규화)의 변화를 하나씩 바꾸어 성능 변화를 관찰합니다.
- 각 실험은 빠르게 돌아가도록 `epochs=5`로 설정되어 있습니다 (데모용). 실제 비교는 더 많은 epoch으로 수행하세요.

In [ ]:
def run_experiments(vary_param, values, base_params, epochs=5):
    results = {}
    for v in values:
        # 하이퍼파라미터 복사 및 변경
        params = base_params.copy()
        params[vary_param] = v
        # 데이터로더 재설정 (배치 크기 변경 시 적용)
        trainloader = DataLoader(trainset, batch_size=params['batch_size'], shuffle=True, num_workers=2)
        testloader = DataLoader(testset, batch_size=256, shuffle=False, num_workers=2)
        model = LinearCIFAR10().to(device)
        optimizer = optim.SGD(model.parameters(), lr=params['lr'], weight_decay=params['weight_decay'])
        criterion = nn.CrossEntropyLoss()
        # 빠른 학습 실행
        accs = []
        for epoch in range(epochs):
            train_one_epoch(model, trainloader, optimizer, criterion, device)
            acc = evaluate(model, testloader, device)
            accs.append(acc)
        results[v] = accs[-1]  # 마지막 epoch의 정확도 사용
        print(f'{vary_param}={v} -> acc={results[v]:.4f}')
    return results

# 기본값 설정(비교 시 고정값)
base_params = {'lr':1e-2, 'batch_size':128, 'weight_decay':0.0}

# 1) Learning rate 변화
lrs = [1e-1, 1e-2, 1e-3]
lr_results = run_experiments('lr', lrs, base_params, epochs=5)

# 2) Batch size 변화
batches = [32, 128, 512]
batch_results = run_experiments('batch_size', batches, base_params, epochs=5)

# 3) Weight decay 변화
wds = [0.0, 1e-4, 1e-3]
wd_results = run_experiments('weight_decay', wds, base_params, epochs=5)

# 결과 요약 시각화
plt.figure(figsize=(12,4))
plt.subplot(1,3,1)
plt.bar([str(x) for x in lrs], [lr_results[x] for x in lrs])
plt.title('LR vs Acc')
plt.subplot(1,3,2)
plt.bar([str(x) for x in batches], [batch_results[x] for x in batches])
plt.title('Batch size vs Acc')
plt.subplot(1,3,3)
plt.bar([str(x) for x in wds], [wd_results[x] for x in wds])
plt.title('Weight decay vs Acc')
plt.show()

## 7) 관찰 및 결론(요약)
- 학습률(LR): 큰 LR(예: 1e-1)은 빠르게 수렴하거나 불안정해질 수 있고, 작은 LR(1e-3)은 안정적이지만 느리게 수렴합니다.
- 배치 크기: 작은 배치는(예: 32) 더 noisy하지만 일반화가 좋아질 수 있고, 큰 배치는(예: 512) 더 안정적이지만 메모리/일반화 측면에서 차이가 있습니다.
- Weight decay: 정규화 효과로 과적합을 줄여주지만, 선형 모델에서는 큰 영향이 아닐 수 있습니다.

권장 실험: epoch을 늘리고(예: 50-100), learning rate schedule(SGD+Momentum, Adam, lr decay)과 배치 크기/augmentations를 조합해 보세요.